# Do Prompts Predict the Score?

The previous notebook asked whether the recorded parameters and the stored
analysis values track the score. This one asks the same question of the prompt
text: given a prompt, can the ranking be anticipated at all?

Two levels of detail, because they answer different questions:

- **tagged** — does using a term raise or lower the score?
- **raw** — does prompt structure (length, weighting, nesting, connectors)
  matter independently of which terms appear?

Every term effect is reported with a significance test and a correction for
running thousands of tests at once, and section 4 shows why the uncorrected
number is misleading.

## Where the numbers come from

| what | source |
|---|---|
| target score | `replay_ratings` over `output/vectors/comparisons.jsonl`, then `public_score_from_rating` |
| prompt text | `positive_prompt` and `negative_prompt` in `output/vectors/text_data.jsonl`, extracted with `extract_prompt_tags` |
| term statistics | `ParameterAnalyzer.analyze_term_correlations`, which owns the term floor and the correction |

Nothing is written except the analyzer's own `output/analysis/term_correlations.json`,
which is the same artifact the CLI produces.

All tqdm progress bars use `delay=3` matching the codebase style.

In [ ]:
import json
import time
from pathlib import Path

import comfyui_image_scorer
import matplotlib.pyplot as plt

# The analysis package is imported as an installed package, so its own location
# is the reliable way back to the config and model artifacts next to it.
PACKAGE_ROOT: Path = Path(comfyui_image_scorer.__file__).resolve().parent
CONFIG_ROOT: Path = PACKAGE_ROOT / "config"
from collections import Counter
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

from comfyui_image_scorer.application.analysis.inputs import AnalysisInputs
from comfyui_image_scorer.application.analysis.parameter_analysis import (
    MIN_TERM_IMAGES,
    ParameterAnalyzer,
)
from comfyui_image_scorer.core.filesystem.paths import text_data_file
from comfyui_image_scorer.core.io.serialization import (
    extract_prompt_tags,
    load_single_jsonl,
)
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)
from comfyui_image_scorer.domain.analysis.derived_features import prompt_features
from comfyui_image_scorer.domain.analysis.trueskill import (
    public_score_from_rating,
    replay_ratings,
)
from comfyui_image_scorer.domain.training.statistics import (
    benjamini_hochberg,
    correlation_with_ci,
    mann_whitney_pvalue,
)
from comfyui_image_scorer.domain.vectors.terms import extract_terms
from comfyui_image_scorer.infrastructure.loading.training_loader import training_loader
from comfyui_image_scorer.domain.data_transformation.data_transformer import (
    get_feature_mapping_from_config,
)
from comfyui_image_scorer.infrastructure.ml_models.plot import PlotManager
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)

SEED: int = 20251015

# Clustering sample. 28k prompts against a few thousand terms makes k-means
# slow and the answer stable well before the full set, so the sample size is
# printed wherever a result depends on it.
CLUSTER_SAMPLE: int = 8000
CLUSTER_MAX: int = 12
# Terms per cluster used for the characteristic-term comparison.
CHARACTERISTIC_TERMS: int = 12
# Images shown in each cluster example grid.
EXAMPLE_GRID: int = 4

# Correlation sample and floor, matching the parameter notebook so the two
# tables of coefficients can be read against each other.
CORRELATION_SAMPLE: int = 10000
MIN_FEATURE_IMAGES: int = 100

## 1. Replay the Comparisons for the Target

The same in-memory replay as the parameter notebook: the rating is recalculated
from the stored comparison order and converted with the module's public score
function, rather than read from a cached `scores.jsonl`.

In [ ]:
comparison_rows = training_loader.load_comparison_rows()
replay_start = time.perf_counter()
replayed = replay_ratings(comparison_rows, order="default")
scores: dict[str, float] = {
    filename: public_score_from_rating(rating)
    for filename, (rating, _count) in replayed.items()
}
logger.info(
    "replayed %d comparisons over %d images in %.1fs",
    len(comparison_rows),
    len(scores),
    time.perf_counter() - replay_start,
)

## 2. What The Training Run Was Allowed To Learn From

Two filters decide what a training run actually sees, and this notebook is
looking at a wider population than that: every recorded comparison, including
the ones too thin to train on.

`comparison_rule.npz` records the comparisons the last run kept and the public
score recomputed on exactly that subset. It is the closest available answer to
"what would training have learned from the kept set alone", so the tables here
are built twice — once on the full replay, once on the rule — and the two are
compared. A result that survives only on the full replay was carried by the thin
tail the filter removes, which makes it a property of the comparison archive
rather than of the model.

`feature_rule.npz` records which columns survived training. A feature dropped
there can still correlate with the score here, but no model ever saw it, so it
cannot be an actionable lever. Both filters are read as recorded state, so this
section reports what the run used rather than re-deriving the rule.

In [ ]:
# --- What the last training run kept, read back as recorded state ---

with (CONFIG_ROOT / "training_config.json").open(encoding="utf-8") as handle:
    training_config = json.load(handle)

min_comparisons: int = training_config["min_comparisons_threshold"]

# filename -> (public score on the kept subset, post-filter comparison count)
rule = training_loader.load_comparison_rule(min_comparisons)

# filename -> public score on the full replay, which is what the rest of this
# notebook measures against.
full_scores: dict[str, float] = dict(scores)
# filename -> public score recomputed on the kept subset only.
rule_scores: dict[str, float] = (
    {filename: value for filename, (value, _count) in rule.items()} if rule else {}
)

logger.info(
    "comparison rule at threshold %d kept %d of %d rated images (%.1f%%)",
    min_comparisons,
    len(rule_scores),
    len(full_scores),
    100.0 * len(rule_scores) / max(len(full_scores), 1),
)
logger.info(
    "images the full replay rates but the rule drops: %d",
    len(set(full_scores) - set(rule_scores)),
)

kept_indices = training_loader.load_feature_rule()
feature_mapping = get_feature_mapping_from_config()
if kept_indices is not None:
    kept_features: dict[str, int] = {}
    for index in kept_indices:
        vector_name = feature_mapping["feature_to_vector"][int(index)]["vector_name"]
        kept_features[vector_name] = kept_features.get(vector_name, 0) + 1
    logger.info(
        "feature rule kept %d of %d columns, spread over %d of %d groups",
        len(kept_indices),
        feature_mapping["total_features"],
        len(kept_features),
        len(feature_mapping["vector_ranges"]),
    )
    for vector_name, kept in sorted(
        kept_features.items(), key=lambda item: (-item[1], item[0])
    )[:12]:
        slot_size = feature_mapping["vector_ranges"][vector_name]["slot_size"]
        logger.info(
            "  %-24s %4d of %4d columns survived (%.0f%%)",
            vector_name,
            kept,
            slot_size,
            100.0 * kept / slot_size,
        )
else:
    logger.info("no feature rule cached")

## 3. Read the Prompts

`text_data.jsonl` stores the prompt in two places per row, and they are not the
same field: `positive_prompt` and `negative_prompt` are what the writer read
from the image metadata, while `prompt_tags` is derived from the positive prompt
alone by `extract_prompt_tags`, which also recurses into nested metadata blocks.

The negative prompt is the part worth watching: it is absent from the database
altogether, so anything that analyses negative prompts is reading the text
sidecar alone. Coverage is measured first, because a term analysis over an empty
column reports nothing and says so silently.

Both are streamed once here rather than re-read per cell; the sidecar holds one
object per scored image and is not cheap to parse.

In [ ]:
rows_started = time.perf_counter()
positive_prompts: dict[str, str] = {}
negative_prompts: dict[str, str] = {}
model_of: dict[str, str] = {}

for row in load_single_jsonl(text_data_file):
    filename, payload = next(iter(row.items()))
    name = str(filename)
    if name not in scores:
        continue
    # extract_prompt_tags handles the wrapped metadata shape some writers emit,
    # where the prompt sits one level down rather than at the top.
    positive = extract_prompt_tags(payload)
    if positive:
        positive_prompts[name] = positive
    negative = payload.get("negative_prompt")
    if isinstance(negative, str) and negative:
        negative_prompts[name] = negative
    weights = payload.get("model") or "{}"
    model_of[name] = max(
        (weight for weight in weights[1:-1].split(",") if ":" in weight),
        key=lambda part: float(part.rsplit(":", 1)[1]),
        default="",
    ).rsplit(":", 1)[0].strip().strip("{}'\"")

logger.info(
    "read %d scored images in %.1fs", len(scores), time.perf_counter() - rows_started
)
logger.info(
    "positive prompts: %d of %d scored images (%.1f%%)",
    len(positive_prompts),
    len(scores),
    100.0 * len(positive_prompts) / len(scores),
)
logger.info(
    "negative prompts: %d of %d scored images (%.1f%%)",
    len(negative_prompts),
    len(scores),
    100.0 * len(negative_prompts) / len(scores),
)

lengths = [len(text) for text in positive_prompts.values()]
logger.info(
    "positive prompt length: median %d chars, p10 %d, p90 %d, max %d",
    int(np.median(lengths)),
    int(np.percentile(lengths, 10)),
    int(np.percentile(lengths, 90)),
    max(lengths),
)
negative_lengths = [len(text) for text in negative_prompts.values()]
if negative_lengths:
    logger.info(
        "negative prompt length: median %d chars, p90 %d, max %d",
        int(np.median(negative_lengths)),
        int(np.percentile(negative_lengths, 90)),
        max(negative_lengths),
    )

# The distinct negatives are few: negative prompts are a stock list, so this
# measures how many distinct strings the dataset actually varies over.
negative_counts = Counter(negative_prompts.values())
logger.info(
    "negative prompts: %d distinct strings across %d images; most common covers %.1f%%",
    len(negative_counts),
    len(negative_prompts),
    100.0 * negative_counts.most_common(1)[0][1] / len(negative_prompts),
)

# The scored images without a positive prompt are absent from every prompt
# section, and their scores are compared here so the exclusion is visible.
without_prompt = {name: score for name, score in scores.items() if name not in positive_prompts}
if without_prompt:
    logger.info(
        "%d images have no positive prompt; their mean public score is %.4f against "
        "%.4f for the rest",
        len(without_prompt),
        float(np.mean(list(without_prompt.values()))),
        float(np.mean([scores[name] for name in positive_prompts])),
    )

## 4. Term Effects, With the Multiple-Comparison Correction

`analyze_term_correlations` extracts weighted terms from both prompts and
reports, for every term seen on at least `MIN_TERM_IMAGES` images:

- `avg_score` — mean public score of the images carrying the term
- `baseline_avg_score` — mean score of the images **not** carrying it
- `delta` — the gap between the two, which is the readable figure
- `p_value` — Mann-Whitney U on the two groups
- `q_value` — that p-value after Benjamini-Hochberg correction
- `cliffs_delta` — effect size, which stays meaningful when the p-value does not

The plain `avg_score` is *not* the effect. A term used on most of the dataset inherits
the dataset average, so the baseline column is what makes the gap readable.

**Reading the charts.** Bars run from the lowest `delta` to the highest, so the
middle is "no effect". A bar whose q-value survives correction is one where the
gap is unlikely to be noise; the count is printed with the chart rather than left
to be inferred from bar length.

`weighted_avg_score` is kept alongside for comparison: it multiplies the score by
the term's weight, which was the original measurement and mixes "this term
appears" with "this term is weighted up".

In [ ]:
inputs = AnalysisInputs(scores=scores, prompt_tags=positive_prompts)
# The analyzer re-reads the sidecar itself; it needs the row stream, not the
# prompts collected above, because it decodes every column it scores against.
analyzer = ParameterAnalyzer(
    inputs, partial(load_single_jsonl, text_data_file)
)
terms_started = time.perf_counter()
summaries = analyzer.analyze_term_correlations()
logger.info("term analysis finished in %.1fs", time.perf_counter() - terms_started)

for polarity, summary in summaries.items():
    logger.info(
        "%s: %d rows, %d distinct terms, %d above the floor of %d, %d dropped",
        polarity,
        summary.rows_analyzed,
        summary.unique_terms,
        summary.kept_terms,
        MIN_TERM_IMAGES,
        summary.dropped_below_floor,
    )

positive_terms = pd.DataFrame(summaries["positive"].terms)
negative_terms = pd.DataFrame(summaries["negative"].terms)
logger.info(
    "positive terms surviving correction: %d of %d",
    int((positive_terms["q_value"] <= 0.05).sum()),
    len(positive_terms),
)

TOP_TERMS: int = 20
strongest = positive_terms.sort_values("delta", ascending=False).head(TOP_TERMS)
weakest = positive_terms.sort_values("delta").head(TOP_TERMS)
logger.info("highest delta terms: %s", ", ".join(strongest["term"].head(8)))
logger.info("lowest delta terms: %s", ", ".join(weakest["term"].head(8)))

# One bar per term, whiskers suppressed: the bar length is the score gap, so a
# whisker would be a second estimate of the same quantity rather than an
# interval on it. The n in each label is the evidence behind the bar.
def _term_bars(terms: pd.DataFrame, polarity: str) -> None:
    ordered = terms.sort_values("delta")
    PlotManager.plot_correlation_bars(
        labels=[
            f"{row.term} (n={row.count})" for row in ordered.itertuples()
        ],
        coefficients=ordered["delta"].tolist(),
        ci_low=ordered["delta"].tolist(),
        ci_high=ordered["delta"].tolist(),
        q_values=ordered["q_value"].tolist(),
        value_label="score gap against images without the term (higher is better)",
        title=(
            f"{polarity} terms ranked by score gap "
            f"({len(terms):,} terms above a floor of {MIN_TERM_IMAGES} images)\n"
            "bar length is the gap; the label's n is the evidence behind it"
        ),
    )


_term_bars(positive_terms, "Positive prompt")

if not negative_terms.empty:
    _term_bars(negative_terms, "Negative prompt")
    logger.info(
        "negative prompt terms: %d survive correction of %d, measured over %d images "
        "carrying a negative prompt",
        int((negative_terms["q_value"] <= 0.05).sum()),
        len(negative_terms),
        len(negative_prompts),
    )

logger.info(
    "strongest term %-28s carries %.4f against %.4f elsewhere, delta %+.4f, q=%.4g",
    strongest.iloc[0]["term"],
    float(strongest.iloc[0]["avg_score"]),
    float(strongest.iloc[0]["baseline_avg_score"]),
    float(strongest.iloc[0]["delta"]),
    float(strongest.iloc[0]["q_value"]),
)
logger.info(
    "term count against |delta|: Spearman %+.3f over %d terms, so the biggest gaps are "
    "not simply the rarest terms",
    float(pd.Series(positive_terms["count"]).corr(pd.Series(positive_terms["delta"]), method="spearman")),
    len(positive_terms),
)

### 4b. The Same Term Gaps On The Comparison Rule

The gaps above are measured against the full replay. Re-running the strongest
terms against the score recomputed on the kept subset alone shows which of them
the filter actually supports.

A term whose gap survives is a property of the data the model trains on. One
that collapses was carried by the thin tail the filter removes, and a model
built on the rule would never have seen that evidence.

The feature rule reported in section 2 is the other half of the same question,
and it gates something different: which stored prompt columns the trained model
could use at all. A term can be real here and still be unreachable by the model
if its embedding column was dropped.

In [ ]:
RULE_COMPARISON_TERMS: int = 25

rule_inputs = AnalysisInputs(scores=rule_scores, prompt_tags=positive_prompts)
rule_analyzer = ParameterAnalyzer(
    rule_inputs, partial(load_single_jsonl, text_data_file)
)
rule_started = time.perf_counter()
rule_summaries = rule_analyzer.analyze_term_correlations()
logger.info("rule term analysis finished in %.1fs", time.perf_counter() - rule_started)

rule_positive = pd.DataFrame(rule_summaries["positive"].terms)
if rule_positive.empty:
    logger.info("the rule target has too few images to measure any term gap")
else:
    # Only the terms strong enough to act on are re-measured, because the point
    # is whether the headline gaps hold, not to re-rank the whole vocabulary on a
    # target that has fewer images behind every one of them.
    ranked = positive_terms.reindex(
        positive_terms["delta"].abs().sort_values(ascending=False).index
    ).head(RULE_COMPARISON_TERMS)
    comparison = ranked.merge(
        rule_positive[["term", "delta", "count", "q_value"]],
        on="term",
        how="inner",
        suffixes=("_full", "_rule"),
    )
    comparison["shift"] = comparison["delta_rule"] - comparison["delta_full"]
    display(comparison.sort_values("delta_full", key=abs, ascending=False).round(4))

    logger.info(
        "re-measured %d of the %d strongest terms on the rule target",
        len(comparison),
        len(ranked),
    )
    logger.info(
        "%d of %d keep their sign, %d survive FDR q <= 0.05 on the rule target",
        int((np.sign(comparison["delta_full"]) == np.sign(comparison["delta_rule"])).sum()),
        len(comparison),
        int((comparison["q_value_rule"] <= 0.05).sum()),
    )
    logger.info(
        "%d of the %d strongest terms fell below the floor on the rule target",
        len(ranked) - len(comparison),
        len(ranked),
    )
    for row in comparison.sort_values("shift", key=abs, ascending=False).head(12).itertuples():
        logger.info(
            "  %-28s full %+.4f (n=%d) -> rule %+.4f (n=%d)  shift %+.4f  q_rule=%.4g",
            row.term,
            row.delta_full,
            row.count_full,
            row.delta_rule,
            row.count_rule,
            row.shift,
            row.q_value_rule,
        )

## 5. The Confound: Terms Travel With Models

A term that appears only under one checkpoint has its `delta` measured against
whatever that checkpoint tends to score, not against the term. The parameter
notebook measures the spread across models on the same replay, and it is worth
comparing that spread against the term gaps below before reading any single
term as a lever.

Two measurements address this:

- **concentration** — the share of a term's images that sit under its most
  common model. The distribution is printed first, because it says how much of
  the vocabulary is entangled with a checkpoint at all.
- **stratified delta** — the gap recomputed *within* that dominant model only,
  so the checkpoint is held constant. The 20 most entangled terms are
  stratified, because those are the ones whose gap is least attributable to the
  term itself.

Each whisker below spans from the unstratified gap to the stratified one. A
sign change is the clearest case: the term looked as though it raised the score,
and raises it not at all once the checkpoint is fixed.

Stratification costs sample size, so a term can fall below the floor on one side
and drop out. That is the honest limit of the check — a term that only ever
appears with one model has no within-model evidence, and no statistical care
manufactures some.

In [ ]:
# term -> Counter(model), built from the prompts already read.
term_models: dict[str, Counter[str]] = {}
images_per_term: Counter[str] = Counter()
image_terms: dict[str, list[str]] = {}

extraction_started = time.perf_counter()
for name, text in positive_prompts.items():
    terms = [term for term, _weight, _index in extract_terms(text).terms]
    image_terms[name] = terms
    images_per_term.update(terms)
    model = model_of.get(name, "")
    for term in terms:
        term_models.setdefault(term, Counter())[model] += 1
logger.info("extracted terms for %d images in %.1fs", len(image_terms), time.perf_counter() - extraction_started)

concentration: dict[str, float] = {}
for term, models in term_models.items():
    total = sum(models.values())
    if total:
        concentration[term] = max(models.values()) / total

eligible = positive_terms[positive_terms["count"] >= MIN_TERM_IMAGES].copy()
eligible["concentration"] = [
    concentration.get(str(term), 0.0) for term in eligible["term"]
]
eligible["term"] = eligible["term"].astype(str)

# The concentration distribution first: it says how much of the vocabulary is
# entangled with a checkpoint at all, which the earlier summary statistic hid.
for threshold in (0.50, 0.70, 0.90, 1.00):
    share = float((eligible["concentration"] >= threshold).mean())
    logger.info(
        "%5.0f%% of the %d ranked terms are used under one model at least %.0f%% of the time",
        100.0 * share,
        len(eligible),
        threshold * 100.0,
    )
logger.info(
    "median concentration %.3f, so most terms appear across several models and "
    "the stratification below is a check on the tail rather than the whole table",
    float(eligible["concentration"].median()),
)

# The most entangled terms are stratified, whether or not they clear a cutoff:
# those are the ones whose gap is least attributable to the term itself.
STRATIFY_COUNT: int = 20
locked = eligible.sort_values("concentration", ascending=False).head(STRATIFY_COUNT)
logger.info(
    "stratifying the %d most model-entangled terms (concentration %.2f to %.2f)",
    len(locked),
    float(locked["concentration"].min()),
    float(locked["concentration"].max()),
)

# Within each term's dominant model, compare the images carrying it against the
# images under the same model that do not.
# Index the images by model once: the filter below runs per term, and scanning
# 28k prompts per term would dominate the runtime.
model_members: dict[str, dict[str, set[str]]] = {}
for name in image_terms:
    model_members.setdefault(model_of.get(name, ""), {})[name] = set(image_terms[name])

stratified_rows: list[dict[str, object]] = []
for row in locked.itertuples():
    term = str(row.term)
    models = term_models.get(term)
    if not models:
        continue
    dominant = models.most_common(1)[0][0]
    members = model_members.get(dominant, {})
    with_term = [
        scores[name] for name, terms in members.items() if term in terms
    ]
    without_term = [
        scores[name] for name, terms in members.items() if term not in terms
    ]
    if len(with_term) < MIN_FEATURE_IMAGES or len(without_term) < MIN_FEATURE_IMAGES:
        continue
    stratified_rows.append(
        {
            "term": term,
            "model": dominant,
            "concentration": float(row.concentration),
            "unstratified_delta": float(row.delta),
            "stratified_delta": float(np.mean(with_term) - np.mean(without_term)),
            "n_within_model": len(with_term),
            "n_without_within_model": len(without_term),
            "p_value": mann_whitney_pvalue(with_term, without_term),
        }
    )

stratified = pd.DataFrame(stratified_rows)
dropped = len(locked) - len(stratified)
if stratified.empty:
    logger.info(
        "none of the %d most entangled terms kept %d images on both sides of its "
        "dominant model",
        len(locked),
        MIN_FEATURE_IMAGES,
    )
else:
    logger.info(
        "stratified %d of the %d most entangled terms; %d dropped for lacking %d "
        "images on one side within their dominant model",
        len(stratified),
        len(locked),
        dropped,
        MIN_FEATURE_IMAGES,
    )
    sign_flips = int(
        (np.sign(stratified["stratified_delta"]) != np.sign(stratified["unstratified_delta"])).sum()
    )
    survived = int((stratified["p_value"] <= 0.05).sum())
    logger.info(
        "%d of %d change sign once the model is held constant, and %d still show a "
        "significant gap within their own model",
        sign_flips,
        len(stratified),
        survived,
    )
    display(stratified.sort_values("unstratified_delta", ascending=False).round(4))

    PlotManager.plot_correlation_bars(
        labels=[
            f"{row.term} on {row.model}"
            for row in stratified.sort_values("unstratified_delta").itertuples()
        ],
        coefficients=stratified.sort_values("unstratified_delta")["unstratified_delta"].tolist(),
        ci_low=stratified.sort_values("unstratified_delta")["stratified_delta"].tolist(),
        ci_high=stratified.sort_values("unstratified_delta")["unstratified_delta"].tolist(),
        q_values=stratified.sort_values("unstratified_delta")["p_value"].tolist(),
        value_label="score gap (higher is better)",
        title=(
            "Single-model terms, unstratified against within-model\n"
            "a bar whose marker sits far from the whisker end means the model "
            "accounted for most of the gap"
        ),
    )

## 6. Raw Prompt Structure

The term section asks which terms. This asks how the prompt is *written*, holding the
vocabulary aside: length, term count, hand-written weights, nesting depth,
connector words, and where terms sit in the sequence.

Each is correlated against the score with the same bootstrap interval and
correction as the parameter notebook, so the two are directly comparable.

Position is measured as the mean of each term's index divided by the term count,
which is scale-free: 0 means every term leads the prompt, 1 means every term
trails it.

In [ ]:
structure_started = time.perf_counter()
structure_records: list[dict[str, object]] = []
for name, text in positive_prompts.items():
    extraction = extract_terms(text)
    record: dict[str, object] = {"filename": name, "score": scores[name]}
    record.update(prompt_features(text, extraction))
    structure_records.append(record)
logger.info(
    "measured prompt structure for %d images in %.1fs",
    len(structure_records),
    time.perf_counter() - structure_started,
)

structure = pd.DataFrame(structure_records)
structure = structure.drop(columns=["filename"])

correlated = structure.sample(
    n=min(CORRELATION_SAMPLE, len(structure)), random_state=SEED
).reset_index(drop=True)
logger.info("correlating prompt structure on %d images", len(correlated))

structure_target = correlated["score"].to_numpy(dtype=float)
structure_rows: list[dict[str, object]] = []
for column in correlated.columns:
    if column == "score":
        continue
    values = correlated[column].to_numpy(dtype=float)
    measured = np.isfinite(values)
    if measured.sum() < MIN_FEATURE_IMAGES:
        continue
    result = correlation_with_ci(
        values[measured], structure_target[measured], "spearman", iterations=200, seed=SEED
    )
    structure_rows.append(
        {
            "measure": column,
            "n": result.sample_count,
            "spearman": result.coefficient,
            "spearman_low": result.ci_low,
            "spearman_high": result.ci_high,
            "p_value": result.p_value,
        }
    )

structure_correlations = pd.DataFrame(structure_rows)
structure_correlations["q_value"] = benjamini_hochberg(
    structure_correlations["p_value"].tolist()
)
display(structure_correlations.sort_values("spearman", key=abs, ascending=False).round(4))

PlotManager.plot_correlation_bars(
    labels=structure_correlations["measure"].tolist(),
    coefficients=structure_correlations["spearman"].tolist(),
    ci_low=structure_correlations["spearman_low"].tolist(),
    ci_high=structure_correlations["spearman_high"].tolist(),
    q_values=structure_correlations["q_value"].tolist(),
    value_label="Spearman rho with public score (either sign)",
    title=(
        f"Prompt structure against score, {len(structure_correlations)} measures\n"
        "whiskers are 95% bootstrap intervals; a bar crossing zero has no "
        "measurable relationship"
    ),
)

logger.info(
    "%d of %d structural measures survive FDR q <= 0.05",
    int((structure_correlations["q_value"] <= 0.05).sum()),
    len(structure_correlations),
)

# Length and term count are near-duplicates by construction, so the pair is
# reported together rather than as two independent findings.
for first, second in (("prompt_chars", "prompt_terms"), ("prompt_mean_weight", "prompt_max_weight")):
    if first in correlated.columns and second in correlated.columns:
        logger.info(
            "%s and %s correlate at rho %+.3f, so they are not independent measures",
            first,
            second,
            float(correlated[first].corr(correlated[second], method="spearman")),
        )

## 7. Prompt Space: Clusters

Terms say *which* words. Clustering the prompts as documents says how many
distinct prompt styles the dataset actually contains, and whether those styles
score differently at all.

The pipeline is TF-IDF over the extracted terms, reduced by truncated SVD to two
dimensions for the picture and to a higher-dimensional space for k-means, so
distance is measured on the vocabulary rather than on the two plotted axes.

`k` is chosen by silhouette score over the range below rather than assumed. The
cluster mean scores are reported with their spread; a cluster whose mean sits
inside another cluster's range has no distinct score to offer.

In [ ]:
documents = [positive_prompts[name] for name in positive_prompts]
filenames = list(positive_prompts)
cluster_indices = np.arange(len(documents))
if len(documents) > CLUSTER_SAMPLE:
    cluster_rng = np.random.default_rng(SEED)
    cluster_indices = np.sort(
        cluster_rng.choice(len(documents), size=CLUSTER_SAMPLE, replace=False)
    )

sampled_documents = [documents[index] for index in cluster_indices]
sampled_scores = np.array([scores[filenames[index]] for index in cluster_indices])

# Terms rather than raw text: the TF-IDF vocabulary is the term vocabulary, so a
# cluster's characteristic terms can be read straight off the same model.
term_documents = [
    " ".join(term for term, _weight, _index in extract_terms(text).terms)
    for text in sampled_documents
]

vectorizer = TfidfVectorizer(min_df=5, max_df=0.6, sublinear_tf=True)
term_matrix = vectorizer.fit_transform(term_documents)
logger.info(
    "TF-IDF over %d sampled prompts: %d documents x %d terms (min_df=5, max_df=0.6)",
    len(sampled_documents),
    term_matrix.shape[0],
    term_matrix.shape[1],
)

components = 50
reducer = TruncatedSVD(n_components=components, random_state=SEED)
reduced = reducer.fit_transform(term_matrix)
logger.info(
    "reduced to %d components explaining %.1f%% of the term variance",
    components,
    100.0 * float(reducer.explained_variance_ratio_.sum()),
)

silhouettes: list[tuple[int, float]] = []
for k in range(2, CLUSTER_MAX + 1):
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(reduced)
    # A silhouette needs at least as many samples as clusters, and a degenerate
    # split scores negative, which is a usable answer rather than an error.
    silhouettes.append((k, float(silhouette_score(reduced, labels))))
for k, value in silhouettes:
    logger.info("  k=%2d silhouette %.4f", k, value)

best_k = max(silhouettes, key=lambda item: item[1])[0]
labels = KMeans(n_clusters=best_k, random_state=SEED, n_init=10).fit_predict(reduced)
logger.info("chosen k=%d on silhouette %.4f", best_k, dict(silhouettes)[best_k])

# The picture is a separate 2-component fit from the clustering space, and its
# variance share is read once rather than refitting inside the label.
two_dim_reducer = TruncatedSVD(n_components=2, random_state=SEED)
two_dim = two_dim_reducer.fit_transform(term_matrix)
two_dim_variance = two_dim_reducer.explained_variance_ratio_

cluster_summary = pd.DataFrame(
    {
        "cluster": labels,
        "score": sampled_scores,
    }
).groupby("cluster")["score"].agg(["count", "mean", "std", "median"])

# A cluster is only worth calling out when its mean sits outside the spread of
# the images outside it.
overall_std = float(sampled_scores.std())
cluster_summary["outside_overall_spread"] = (
    (cluster_summary["mean"] - float(sampled_scores.mean())).abs() > overall_std
)
logger.info(
    "cluster mean scores span %.4f to %.4f against an overall sd of %.4f; "
    "%d of %d cluster means sit more than one sd from the dataset mean",
    float(cluster_summary["mean"].min()),
    float(cluster_summary["mean"].max()),
    overall_std,
    int(cluster_summary["outside_overall_spread"].sum()),
    len(cluster_summary),
)
display(cluster_summary.round(4))

figure, axis = plt.subplots(figsize=(12, 8))
scatter = axis.scatter(
    two_dim[:, 0], two_dim[:, 1], c=sampled_scores, s=4, alpha=0.4, cmap="viridis"
)
axis.set_xlabel(
    f"SVD component 1 ({100.0 * float(two_dim_variance[0]):.1f}% of variance)"
)
axis.set_ylabel(
    f"SVD component 2 ({100.0 * float(two_dim_variance[1]):.1f}% of variance)"
)
axis.set_title(
    f"Prompt space, {len(sampled_documents):,} sampled prompts, colour = public score\n"
    f"the {best_k}-means partition is fitted on {components} components, not on these two"
)
figure.colorbar(scatter, ax=axis, label="public score")
axis.grid(linestyle=":", alpha=0.4)
figure.tight_layout()
plt.show()

## 8. What Each Cluster Is Made Of

A cluster only means something if its members share terms. Each cluster is
compared against the rest of the sample by log-odds on term presence, which is
unaffected by how common a term is overall — a plain frequency ratio would rank
the most ubiquitous term highest in every cluster.

Positive log-odds means the term is over-represented in that cluster. The two
columns are the measure and the count, because a term appearing twice should not
be read as strongly as one appearing a thousand times.

The example grids show real images from the highest and lowest scoring clusters,
which is where the cluster labels either match what is in the pictures or
obviously do not.

In [ ]:
binary = (term_matrix > 0).astype(np.int8)
presence = binary.toarray()
cluster_ids = sorted(set(labels.tolist()))
global_rate = presence.mean(axis=0)
log_odds_rows: list[dict[str, object]] = []

for cluster_id in cluster_ids:
    member = presence[labels == cluster_id]
    member_rate = member.mean(axis=0)
    # Laplace smoothing: a term absent from a cluster would otherwise divide by
    # zero, and a rare term present in one image would look decisive.
    smoothed = (member.sum(axis=0) + 0.5) / (member.shape[0] + 1.0)
    baseline = (presence.sum(axis=0) + 0.5) / (presence.shape[0] + 1.0)
    effect = np.log(smoothed / baseline)
    keep = np.argsort(effect)[::-1][:CHARACTERISTIC_TERMS]
    for column in keep:
        log_odds_rows.append(
            {
                "cluster": cluster_id,
                "term": vectorizer.get_feature_names_out()[column],
                "log_odds": float(effect[column]),
                "images": int(member[:, column].sum()),
                "share_in_cluster": float(global_rate[column]),
            }
        )

characteristics = pd.DataFrame(log_odds_rows)
display(characteristics.head(30).round(4))

for cluster_id in cluster_ids:
    subset = characteristics[characteristics["cluster"] == cluster_id].head(6)
    logger.info(
        "cluster %2d (n=%d, mean score %.4f): %s",
        cluster_id,
        int((labels == cluster_id).sum()),
        float(sampled_scores[labels == cluster_id].mean()),
        ", ".join(f"{row.term}" for row in subset.itertuples()),
    )


def _resolve(filename: str) -> str | None:
    path = find_image_path(filename)
    return str(path) if path is not None else None


# Cluster means, not per-image scores: the extreme single image is a prompt with
# an unusual combination, which is not what "which cluster scores highest" means.
cluster_mean_scores = {
    int(cluster_id): float(sampled_scores[labels == cluster_id].mean())
    for cluster_id in cluster_ids
}
best_cluster = max(cluster_mean_scores, key=cluster_mean_scores.get)
worst_cluster = min(cluster_mean_scores, key=cluster_mean_scores.get)
logger.info(
    "highest scoring cluster %d (mean %.4f) against lowest %d (mean %.4f)",
    best_cluster,
    cluster_mean_scores[best_cluster],
    worst_cluster,
    cluster_mean_scores[worst_cluster],
)

for cluster_id, label in ((best_cluster, "highest"), (worst_cluster, "lowest")):
    member_positions = np.flatnonzero(labels == cluster_id)
    member_scores = sampled_scores[member_positions]
    if member_scores.size == 0:
        continue
    order = member_positions[np.argsort(member_scores)]
    # The highest cluster shows both tails: the point is how far apart the two
    # ends of one cluster sit, not that its worst member is low.
    if label == "highest" and member_scores.size > EXAMPLE_GRID * 2:
        chosen = np.concatenate([order[-EXAMPLE_GRID:], order[:EXAMPLE_GRID]])
    else:
        chosen = order[:EXAMPLE_GRID]
    PlotManager.plot_image_grid(
        filenames=[filenames[cluster_indices[position]] for position in chosen],
        titles=[
            f"cluster {cluster_id}\nscore {sampled_scores[position]:.3f}"
            for position in chosen
        ],
        path_for=_resolve,
        columns=EXAMPLE_GRID,
        title=f"{label.title()} scoring images from prompt cluster {cluster_id}",
    )
    logger.info(
        "cluster %d spans %.4f to %.4f across %d sampled prompts",
        cluster_id,
        float(member_scores.min()),
        float(member_scores.max()),
        len(member_scores),
    )

## 9. What This Does Not Show

- **No causation.** A term's `delta` is the score gap between images that use it
  and images that do not. Prompt length, checkpoint, LoRA and content all move
  together in a generated dataset.
- **The images without a positive prompt** are excluded from every prompt
  section. Their mean score is printed where the prompts are read, so the
  exclusion is visible.
- **Negative prompts are a stock list.** The prompt section prints the distinct-string
  count; the term analysis over them measures the vocabulary, not per-image
  choices.
- **Term deltas are not model-adjusted by default.** The `delta` reported by the
  term section is the raw gap. The confound section prints how concentrated the
  median term is under a single checkpoint, and the most entangled terms still
  need the stratified gap rather than the raw one.
- **The cluster `k` is a choice within the searched range.** Silhouette scores
  are printed for every `k`, so the shape of that choice is visible.